# Portfolio risk: correlated returns and input data

In Radiant: **Load example > Correlated stock returns**, press _Run simulation_ (saved as `returns`), then **Load example > Portfolio weights (input data and sdw)**.

## Question

> Simulate 1,000 monthly returns for three stocks with means 1%, 1.5% and 0.8%, standard deviations 5%, 8% and 3%, and correlations 0.3 (A-B), 0.1 (A-C) and 0.2 (B-C). Save the result as 'returns' to use it as input data for the portfolio example.

In [1]:
import pyrsm as rsm

**Correlations** between normal variables are given per pair. With `normal_exact` the simulated returns have exactly the means, standard deviations and correlations given (radiant.model's "Use exact specifications").

In [2]:
spec = {
    "name": "returns",
    "runs": 1000,
    "seed": 1234,
    "variables": [
        {"name": "A", "kind": "normal", "mean": 0.01, "sd": 0.05},
        {"name": "B", "kind": "normal", "mean": 0.015, "sd": 0.08},
        {"name": "C", "kind": "normal", "mean": 0.008, "sd": 0.03},
    ],
    "normal_exact": True,
    "correlations": [
        {"a": "A", "b": "B", "r": 0.3},
        {"a": "A", "b": "C", "r": 0.1},
        {"a": "B", "b": "C", "r": 0.2},
    ],
}

In [3]:
returns = rsm.decide.simulate(spec).data
returns.corr()

shape: (3, 3)
┌─────┬─────┬─────┐
│ A   ┆ B   ┆ C   │
│ --- ┆ --- ┆ --- │
│ f64 ┆ f64 ┆ f64 │
╞═════╪═════╪═════╡
│ 1.0 ┆ 0.3 ┆ 0.1 │
│ 0.3 ┆ 1.0 ┆ 0.2 │
│ 0.1 ┆ 0.2 ┆ 1.0 │
└─────┴─────┴─────┘

## Question

> Using the simulated 'returns' as input data, which weights for stocks A and B (C gets the rest) give the lowest risk (standard deviation of the portfolio return)? Search weights from 0 to 1 in steps of 0.05.

The simulated returns are the **input data**: their columns (`A`, `B`, `C`) can be used in the formulas but are not part of the result. A grid search over the weights of A and B gives 441 rows; `sdw(wA, wB, wC, A, B, C)` is, for each row of weights, the standard deviation of the weighted sum of the returns.

In [4]:
formulas = """
wC = 1 - wA - wB
valid = wC >= 0
risk = ifelse(valid, sdw(wA, wB, wC, A, B, C), 99)
expected = wA * mean(A) + wB * mean(B) + wC * mean(C)
min_risk_A = find_min(risk, wA)
min_risk_B = find_min(risk, wB)
"""

pspec = {
    "name": "portfolio",
    "runs": 1000,
    "seed": 1234,
    "variables": [
        {"name": "wA", "kind": "grid", "min": 0, "max": 1, "step": 0.05},
        {"name": "wB", "kind": "grid", "min": 0, "max": 1, "step": 0.05},
    ],
    "data": "returns",
    "form": formulas,
}

In [5]:
port = rsm.decide.simulate(pspec, data=returns)
port.summary(dec=3)

Simulation
Simulations: 441
Random seed: 1234
Sim data   : portfolio
Grid search: wA 0 1 0.05; wB 0 1 0.05
Data       : returns
Formulas   :
	wC = 1 - wA - wB
	valid = wC >= 0
	risk = ifelse(valid, sdw(wA, wB, wC, A, B, C), 99)
	expected = wA * mean(A) + wB * mean(B) + wC * mean(C)
	min_risk_A = find_min(risk, wA)
	min_risk_B = find_min(risk, wB)

Constants:
 min_risk_A min_risk_B
      0.250      0.000

Variables:
          n_obs   mean     sd    min    p25 median    p75    max
 wA         441  0.500  0.303  0.000  0.250  0.500  0.750  1.000
 wB         441  0.500  0.303  0.000  0.250  0.500  0.750  1.000
 wC         441 -0.000  0.429 -1.000 -0.300 -0.000  0.300  1.000
 risk       441 50.082 49.532  0.027  0.041 99.000 99.000 99.000
 expected   441  0.012  0.002  0.008  0.011  0.013  0.014  0.017

Logicals:
       TRUE (nr) TRUE (prop)
 valid       218       0.494


In [6]:
import polars as pl

port.data.filter(pl.col("valid")).sort("risk").head(5)

shape: (5, 8)
┌──────┬──────┬──────┬───────┬──────────┬──────────┬────────────┬────────────┐
│ wA   ┆ wB   ┆ wC   ┆ valid ┆ risk     ┆ expected ┆ min_risk_A ┆ min_risk_B │
│ ---  ┆ ---  ┆ ---  ┆ ---   ┆ ---      ┆ ---      ┆ ---        ┆ ---        │
│ f64  ┆ f64  ┆ f64  ┆ bool  ┆ f64      ┆ f64      ┆ f64        ┆ f64        │
╞══════╪══════╪══════╪═══════╪══════════╪══════════╪════════════╪════════════╡
│ 0.25 ┆ 0.0  ┆ 0.75 ┆ true  ┆ 0.02681  ┆ 0.0085   ┆ 0.25       ┆ 0.0        │
│ 0.2  ┆ 0.0  ┆ 0.8  ┆ true  ┆ 0.026907 ┆ 0.0084   ┆ 0.25       ┆ 0.0        │
│ 0.2  ┆ 0.05 ┆ 0.75 ┆ true  ┆ 0.026968 ┆ 0.00875  ┆ 0.25       ┆ 0.0        │
│ 0.3  ┆ 0.0  ┆ 0.7  ┆ true  ┆ 0.027    ┆ 0.0086   ┆ 0.25       ┆ 0.0        │
│ 0.25 ┆ 0.05 ┆ 0.7  ┆ true  ┆ 0.027006 ┆ 0.00885  ┆ 0.25       ┆ 0.0        │
└──────┴──────┴──────┴───────┴──────────┴──────────┴────────────┴────────────┘